# Aura Inference Benchmarks

**Hands-on inference lab**: Deploy a model, serve inference, measure TTFT, KV cache, batching, quantization, and prefix caching.

**Requirements**: Google Colab with T4 GPU (free tier works) or any NVIDIA GPU with 16GB+ VRAM.

---

## Setup

Install dependencies and clone the repo.

**What's happening under the hood:**
- vLLM is an inference engine that uses **PagedAttention** to efficiently manage GPU memory
- The `openai` package lets us talk to vLLM using the same API format as OpenAI/Anthropic
- `aiohttp` enables sending multiple concurrent requests (needed for batching tests in Lab 3)

In [ ]:
# LEARNING: Check what GPU you have and how much VRAM is available.
# The T4 has 15,360 MiB (15 GB) — model weights + KV cache must fit in this.
# Right now usage should be ~0 MiB. Compare this to AFTER model loading.
!nvidia-smi

In [ ]:
# Install vLLM and dependencies into the SAME Python the notebook kernel uses.
# Using {sys.executable} -m pip ensures packages go to the right environment.
import sys
!{sys.executable} -m pip install vllm openai aiohttp matplotlib tabulate numpy pynvml pyyaml tqdm -q

In [ ]:
# Clone the repo
!git clone https://github.com/sshodhan/AuraInferenceBenchmarks.git 2>/dev/null || echo "Already cloned"
import os
os.chdir("AuraInferenceBenchmarks")
!pwd

## Start vLLM Server

We start the server in the background so we can run benchmarks in this notebook.

**Key concept — What happens when a model loads:**
1. **Weights are read from disk** (downloaded from HuggingFace the first time)
2. **Weights are transferred to GPU memory** — this is why loading takes minutes
3. **KV cache space is pre-allocated** — vLLM reserves remaining GPU memory for the KV cache
4. The server is now ready to accept requests via an OpenAI-compatible API

**Watch for:** How long does loading take? Qwen2.5-1.5B is ~3 GB in FP16. On a T4 this typically takes 1-3 minutes.

In [ ]:
import subprocess, time, sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000

# Start vLLM in background
# NOTE: Use sys.executable to ensure we use the same Python where vLLM was installed.
# Using "python" directly can pick up the system Python which won't have vLLM.
server_proc = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", str(PORT)],
    stdout=open("/tmp/vllm_stdout.log", "w"),
    stderr=open("/tmp/vllm_stderr.log", "w"),
)
print(f"Started vLLM server (PID: {server_proc.pid})")
print(f"Model: {MODEL}")
print(f"Loading model... (this takes 1-3 minutes on Colab T4)")

In [ ]:
# Wait for server to be ready
from benchmarks.utils import wait_for_server

ready = wait_for_server(f"http://localhost:{PORT}/v1", timeout=300)
if ready:
    print("Server is ready! Proceeding to labs.")
else:
    print("Server failed to start. Check logs:")
    !tail -20 /tmp/vllm_stderr.log

In [ ]:
# LEARNING: Compare this to the FIRST nvidia-smi output above.
# Notice how "Memory-Usage" jumped from ~0 MiB to ~3000+ MiB.
# That's the model weights now sitting on the GPU.
# The REMAINING memory (~12 GB) is available for KV cache — this determines
# how many concurrent requests and how long prompts can be.
!nvidia-smi

---
## Lab 1: Deploy Your First Model

Send test prompts and measure TTFT + generation time.

**Key concepts to watch:**
- **TTFT (Time to First Token)**: How long before the model starts responding. This is the "prefill" phase — the model processes your entire prompt before generating the first output token.
- **Tokens/sec (TPS)**: How fast the model generates output tokens. This is the "decode" phase — each new token requires one forward pass.
- **Short vs long prompts**: TTFT should be noticeably higher for longer prompts because the prefill phase processes more tokens.

**AuraApp connection:** When the nightly pipeline sends a request with a long system prompt (style rules + city context), that prefill cost is the TTFT you're measuring here.

In [ ]:
from benchmarks.lab1_deploy import run_lab1
run_lab1()

### Lab 1 Reflection

**Write down your observations:**
- TTFT for short prompt: ___ ms → TTFT for long prompt: ___ ms (how much did it increase?)
- Tokens/sec: ___ → is the decode phase fast or slow on T4?
- GPU memory used by the model: ___ MiB out of 15,360 MiB

**Why this matters:** TTFT is what users *feel* as latency. When someone types in Claude or AuraApp, TTFT is the pause before text starts appearing. Faster TTFT = better user experience.

---
## Lab 2: Measure KV Cache Impact

Send prompts of increasing length (100 → 4000 tokens) and watch TTFT + memory scale.

**Key concepts to watch:**
- **KV cache**: For every token in your prompt, the model stores Key and Value vectors in GPU memory. Longer prompt = more KV cache = more memory.
- **Scaling behavior**: Does TTFT grow linearly (2x prompt = 2x TTFT) or faster? Attention is O(n²), so you may see super-linear scaling at longer contexts.
- **Memory climbing**: Watch GPU memory increase as prompt length grows — this is KV cache in action.

**AuraApp connection:** The nightly pipeline uses long system prompts with style rules. If those prompts are 2000 tokens, the KV cache cost per request limits how many requests you can batch together.

In [ ]:
from benchmarks.lab2_kvcache import run_lab2
run_lab2()

In [ ]:
# View the plots
from IPython.display import Image, display
display(Image(filename="results/lab2_ttft_vs_prompt_length.png"))
display(Image(filename="results/lab2_memory_vs_prompt_length.png"))

### Lab 2 Reflection

**Write down your observations:**
- TTFT at 100 tokens: ___ ms → TTFT at 4000 tokens: ___ ms (___x increase)
- Memory at 100 tokens: ___ MiB → Memory at 4000 tokens: ___ MiB
- Does TTFT scale linearly or super-linearly?

**Key insight:** At some prompt length, you'd run out of GPU memory entirely. This is why production systems carefully manage context window sizes — every token costs real GPU memory.

---
## Lab 3: Batching & Throughput

Load test at different concurrency levels to see the throughput-latency tradeoff.

**Key concepts to watch:**
- **Continuous batching**: vLLM batches multiple requests together on the GPU. More requests = better GPU utilization = higher total throughput.
- **Throughput vs latency tradeoff**: Total tokens/sec across ALL requests goes UP with concurrency, but each INDIVIDUAL request gets slower. This is the fundamental tradeoff in inference serving.
- **GPU utilization**: At concurrency=1, the GPU is underutilized during decode (one token at a time). At higher concurrency, the GPU processes multiple requests' tokens in parallel.
- **Saturation point**: At some concurrency level, the GPU or memory is fully utilized — adding more requests just increases latency without improving throughput.

**AuraApp connection:** The nightly pipeline generates 40-60K outfits. Running them one at a time wastes GPU capacity. Batching at the right concurrency level minimizes cost (GPU-hours) while keeping latency acceptable.

In [ ]:
from benchmarks.lab3_batching import run_lab3
run_lab3()

In [ ]:
# View the plots
from IPython.display import Image, display
display(Image(filename="results/lab3_concurrency_vs_throughput.png"))
display(Image(filename="results/lab3_concurrency_vs_per_request_tps.png"))

### Lab 3 Reflection

**Write down your observations:**
- Throughput at concurrency=1: ___ tok/s → concurrency=16: ___ tok/s (___x improvement)
- Per-request TTFT at concurrency=1: ___ ms → concurrency=16: ___ ms (___x slower)
- At what concurrency did throughput plateau? ___
- GPU utilization at concurrency=1: ___% → concurrency=16: ___%

**The interview answer:** "I load-tested inference at varying concurrency levels and observed the throughput-latency tradeoff — throughput plateaued at [X] concurrent requests when GPU memory became the bottleneck."

---
## Lab 4: Compare Model Sizes

**Note**: This lab stops the current server and starts new ones for each model. It takes longer (~5-10 min total).

**Key concepts to watch:**
- **0.5B vs 1.5B parameters**: Larger model = more weights on GPU = less room for KV cache = fewer concurrent users
- **Speed vs quality**: Smaller models are faster but less capable. This is exactly the Haiku/Sonnet/Opus tradeoff at Anthropic.
- **Memory footprint**: 0.5B uses ~1 GB, 1.5B uses ~3 GB. That 2 GB difference means thousands more KV cache slots for concurrent requests.

**AuraApp connection:** For real-time user requests, a small fast model (like Haiku) might be best. For the nightly batch pipeline where quality matters more, a larger model could be worth the slower speed.

In [ ]:
# Stop the current server first
server_proc.terminate()
server_proc.wait()
print("Server stopped.")
time.sleep(5)

In [ ]:
from benchmarks.lab4_model_comparison import run_lab4
import sys
sys.argv = ["lab4"]  # reset argv for argparse
run_lab4()

In [ ]:
# View comparison plots
from IPython.display import Image, display
display(Image(filename="results/lab4_ttft_comparison.png"))
display(Image(filename="results/lab4_memory_comparison.png"))

### Lab 4 Reflection

**Write down your observations:**
- 0.5B TTFT at 1000 tokens: ___ ms → 1.5B TTFT at 1000 tokens: ___ ms
- 0.5B GPU memory: ___ MiB → 1.5B GPU memory: ___ MiB
- 0.5B throughput: ___ tok/s → 1.5B throughput: ___ tok/s

**Key insight:** Larger model = more memory for weights = less room for KV cache = fewer concurrent users. This maps directly to Anthropic's model tiers: Haiku (fast, cheap, smaller) vs Opus (capable, expensive, larger).

---
## Lab 5: Quantization Impact

Compare FP16 vs INT8 (AWQ) — memory savings, speed changes, quality differences.

**Key concepts to watch:**
- **Quantization**: Reducing weight precision from 16-bit to 8-bit (or 4-bit). Each weight takes half the memory.
- **Memory savings**: Should see roughly 40-50% reduction in model memory — this frees up space for more KV cache.
- **Speed impact**: Less data to move through memory bandwidth → potentially faster inference.
- **Quality tradeoff**: The same prompt may produce slightly different (sometimes worse) outputs. Look at the side-by-side comparisons.

**AuraApp connection:** Quantizing the nightly pipeline model to INT8 could cut GPU costs by ~50% with minimal quality impact. This is a real decision inference teams make.

In [ ]:
from benchmarks.lab5_quantization import run_lab5
sys.argv = ["lab5"]  # reset argv
run_lab5()

In [ ]:
# View comparison
from IPython.display import Image, display
try:
    display(Image(filename="results/lab5_quantization_comparison.png"))
except FileNotFoundError:
    print("Plot not generated (only available when both modes are tested)")

### Lab 5 Reflection

**Write down your observations:**
- FP16 memory: ___ MiB → INT8 memory: ___ MiB (___% reduction)
- FP16 TTFT: ___ ms → INT8 TTFT: ___ ms
- FP16 throughput: ___ tok/s → INT8 throughput: ___ tok/s
- Quality difference: noticeable / barely noticeable / none

**The interview answer:** "I compared FP16 vs INT8 quantization and saw [X]% memory reduction with minimal quality impact — this is the kind of tradeoff I'd help the team evaluate for production."

---
## Lab 6: Prompt Caching Simulation

Test prefix caching with a shared long system prompt across multiple requests.

**Key concepts to watch:**
- **Prefix caching**: When multiple requests share the same system prompt, the KV cache for that prefix can be computed once and reused. Subsequent requests skip the redundant prefill.
- **First request vs subsequent**: The first request computes the full KV cache. Requests 2-10 should have dramatically lower TTFT because they reuse the cached prefix.
- **With vs without caching**: This lab tests both modes so you can see the exact TTFT difference.

**AuraApp connection:** The nightly pipeline sends 40-60K requests with the SAME style rules as the system prompt. With prefix caching, you compute those style rules ONCE instead of 60,000 times. This is also exactly how Claude Code works — same codebase context, different questions.

In [ ]:
from benchmarks.lab6_prefix_caching import run_lab6
sys.argv = ["lab6", "--manage-servers"]  # test both modes
run_lab6()

In [ ]:
# View caching results
from IPython.display import Image, display
try:
    display(Image(filename="results/lab6_prefix_caching_ttft.png"))
    display(Image(filename="results/lab6_prefix_caching_bars.png"))
except FileNotFoundError:
    try:
        display(Image(filename="results/lab6_ttft_trend.png"))
    except FileNotFoundError:
        print("No plots generated")

### Lab 6 Reflection

**Write down your observations:**
- First request TTFT (cache cold): ___ ms
- Subsequent request avg TTFT (cache warm): ___ ms
- TTFT improvement: ___% reduction
- Without caching avg TTFT: ___ ms → With caching avg TTFT: ___ ms

**The interview answer:** "I tested prefix caching and saw TTFT drop by [X]% for repeated contexts — this directly maps to serving patterns like Claude Code where users iterate on the same codebase."

**Why this is the most important lab for AuraApp:** The nightly pipeline is 40-60K requests with shared style rules. Prefix caching turns 60,000 full prefills into 1 full prefill + 59,999 cache hits. At 2000-token system prompts, that could save hours of GPU time per night.

---
## Summary & Interview Talking Points

In [ ]:
import json, glob

print("=" * 60)
print("  ALL LAB RESULTS")
print("=" * 60)

for result_file in sorted(glob.glob("results/lab*_results.json")):
    with open(result_file) as f:
        data = json.load(f)
    print(f"\n--- {data.get('lab', result_file)} ---")
    print(json.dumps({k: v for k, v in data.items() if k != 'raw'}, indent=2, default=str)[:500])

print("\n" + "=" * 60)
print("  Fill in your interview talking points:")
print("=" * 60)
print("""
1. "I deployed a model with vLLM and measured TTFT scaling from 100 to 4000 tokens —
    I saw firsthand how KV cache pressure drives memory usage."

2. "I load-tested inference at varying concurrency levels and observed the throughput-latency
    tradeoff — throughput plateaued at [X] concurrent requests when GPU became the bottleneck."

3. "I compared FP16 vs INT8 quantization and saw [X]% memory reduction with minimal quality
    impact — this is the kind of tradeoff I'd help the team evaluate for production."

4. "I tested prefix caching and saw TTFT drop by [X]% for repeated contexts — this directly
    maps to serving patterns like Claude Code where users iterate on the same codebase."
""")

---
## Cleanup

In [ ]:
# Kill any remaining vLLM servers
!pkill -f "vllm.entrypoints" 2>/dev/null || echo "No server to stop"
print("Done! Check the results/ directory for all saved data and plots.")